# CNN training file

In [2]:
# for jupyter notebook
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
import os

# train test split
from sklearn.model_selection import train_test_split

# tensorflow
import tensorflow as tf
import keras

# keras stuff
from keras.datasets import mnist
from keras.models import Model
from keras.layers import Dense, Input, Conv2D, MaxPooling2D, Flatten, Dropout
from keras.optimizers import Adam, SGD
from keras.losses import sparse_categorical_crossentropy, binary_crossentropy

# model selection and stucture
1 lightweight model to place an indoor image to one of the following categories: working_spacecs, store, public_spaces, leisure, home  

Then we will have 5 models for each of these 5 categories to assign them to even more specific categories.  

The training will be done only on a fraction of the images for now for the sake of reducing training times  

In [4]:
# for Le Fan's google drive
working_dir = 'drive/MyDrive/DS4420 project/'
images_dir = 'drive/MyDrive/DS4420 project/Images'
print(os.listdir(images_dir))
# check GPU
tf.config.list_physical_devices('GPU')

os.curdir()

['store', 'home', 'public_spaces', 'leisure', 'working_spaces']


[]

In [5]:
# # preprocessing file will change the dataset under /Images into 5 categories from the MIT dataset, use this if running in local files, not jupyter notebook
!python data_preprocessing.py

python3: can't open file '/content/data_preprocessing.py': [Errno 2] No such file or directory


In [6]:
def folder_to_numpy(folder_path, target_size=(128, 128)):
    images = []

    for i, subfolder in enumerate(os.listdir(folder_path)):
      print(f'Subfolder {i+1}/{len(os.listdir(folder_path))}')
      subfolder_path = os.path.join(folder_path, subfolder)

      for fname in os.listdir(subfolder_path):
        fpath = os.path.join(subfolder_path, fname)

        with Image.open(fpath) as img:
                img = img.convert('L')
                img = ImageOps.pad(img, target_size, color=0)
                images.append(np.array(img, dtype=np.uint8))

    return np.array(images, dtype=np.uint8)

In [7]:
scene_folders = os.listdir(images_dir)
all_data = []
all_labels = []

for label, folder_name in enumerate(scene_folders):
  print(f'Folder {label+1}/{len(scene_folders)}')
  folder_path = os.path.join(images_dir, folder_name)
  data = folder_to_numpy(folder_path)
  labels = np.full(data.shape[0], label)
  all_data.append(data)
  all_labels.append(labels)

indoor_data = np.concatenate(all_data, axis=0)
y = np.concatenate(all_labels, axis=0)

Folder 1/5
Subfolder 1/12


KeyboardInterrupt: 

For our initial model, we took in an image and classified it to one out of 67 classes, now we want to classify it into one out of 5 broader classes

In [ ]:
#inpx = Input(shape=(540, 499, 1))
inpx = Input(shape=(128, 128, 1))

#con_layer = Conv2D(1, kernel_size=(4, 4), strides=2, activation=None, padding='same')(inpx)
con_layer = Conv2D(32, kernel_size=(3,3), activation='relu', padding='same')(inpx)
pool_layer = MaxPooling2D(pool_size=(2,2))(con_layer)

con_layer2 = Conv2D(64, kernel_size=(3,3), activation='relu', padding='same')(pool_layer)
pool_layer2 = MaxPooling2D(pool_size=(2,2))(con_layer2)

con_layer3 = Conv2D(128, kernel_size=(3,3), activation='relu', padding='same')(pool_layer2)
pool_layer3 = MaxPooling2D(pool_size=(2,2))(con_layer3)

flat_G = Flatten()(pool_layer3)
hid_layer = Dense(256, activation='relu')(flat_G)
dropout = Dropout(0.5)(hid_layer)
hid_layer2 = Dense(128, activation='relu')(dropout)
dropout2 = Dropout(0.5)(hid_layer2)
out_layer = Dense(5, activation='softmax')(dropout2)